In [3]:
import json
from kafka import KafkaConsumer

BOOTSTRAP_SERVERS = "localhost:9092,localhost:9094,localhost:9095"
TOPIC = "flights"
GROUP_ID = "flight-notifier"

# Two fleet groups, matching the same pooling decision used for the Spark
# model: B737-NG and B737F-NG share the same airframe and overlapping
# weight range, so they're combined here too.
FLEET_GROUP_MAP = {
    "A330": "A330",
    "B737-NG": "B737",
    "B737F-NG": "B737",
}

FLEET_THRESHOLDS = {
    "A330": {"dist_high": 3567 - 0.10 * (3567 - 136),
             "weight_high": 228376 - 0.10 * (228376 - 137931)},
    "B737": {"dist_high": 2589 - 0.10 * (2589 - 44),
             "weight_high": 76961 - 0.10 * (76961 - 47392)},
}

def check_flight(payload):
    fleet = payload.get("fleet")
    distance = payload.get("great_circle_distance_nm")
    weight = payload.get("gross_weight_at_liftoff_kg")

    group = FLEET_GROUP_MAP.get(fleet)
    thresholds = FLEET_THRESHOLDS.get(group)
    if not thresholds or distance is None or weight is None:
        return None

    reasons = []
    if distance >= thresholds["dist_high"]:
        reasons.append(f"long-haul ({distance:.0f} NM, top 10% for {fleet})")
    if weight >= thresholds["weight_high"]:
        reasons.append(f"heavy ({weight:.0f} kg, top 10% for {fleet})")

    if reasons:
        return f"[NOTABLE] {fleet} flight {payload.get('flight_id', '?')[:8]}... — " + ", ".join(reasons)
    return None


consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=BOOTSTRAP_SERVERS,
    group_id=GROUP_ID,
    auto_offset_reset="latest",
    enable_auto_commit=True,
    auto_commit_interval_ms=5000,
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
)

print(f"Consumer started. Group: {GROUP_ID}")
print("Waiting for partition assignment...")

seen, notified = 0, 0
try:
    for message in consumer:
        assigned = consumer.assignment()
        partitions = sorted(p.partition for p in assigned) if assigned else []

        record = message.value
        payload = record.get("payload", record)

        seen += 1
        note = check_flight(payload)
        if note:
            notified += 1
            print(note)
        else:
            print(f"  ok: {payload.get('fleet', '?')} flight, partitions owned: {partitions}")

        if seen % 10 == 0:
            print(f"--- {seen} flights seen, {notified} flagged, owning partitions {partitions} ---")

except KeyboardInterrupt:
    print(f"\nStopped. Total seen: {seen}, flagged: {notified}")
finally:
    consumer.close()

C:\Users\hp\AppData\Local\Temp\ipykernel_20808\1269930480.py:45: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Consumer started. Group: flight-notifier
Waiting for partition assignment...
[NOTABLE] B737F-NG flight 8a0ed935... — long-haul (2487 NM, top 10% for B737F-NG)
  ok: B737F-NG flight, partitions owned: [0, 1, 2]
  ok: B737F-NG flight, partitions owned: [0, 1, 2]
  ok: B737F-NG flight, partitions owned: [0, 1, 2]
[NOTABLE] A330 flight c9c28e1c... — heavy (224294 kg, top 10% for A330)
  ok: B737F-NG flight, partitions owned: [0, 1, 2]
  ok: B737F-NG flight, partitions owned: [0, 1, 2]
  ok: B737-NG flight, partitions owned: [0, 1, 2]
  ok: A330 flight, partitions owned: [0, 1, 2]
  ok: B737-NG flight, partitions owned: [0, 1, 2]
--- 10 flights seen, 2 flagged, owning partitions [0, 1, 2] ---
  ok: A330 flight, partitions owned: [0, 1, 2]
  ok: B737F-NG flight, partitions owned: [0, 1, 2]
  ok: B737F-NG flight, partitions owned: [0, 1, 2]
[NOTABLE] A330 flight ea37e471... — long-haul (3248 NM, top 10% for A330)
  ok: B737-NG flight, partitions owned: [0, 1, 2]
  ok: B737F-NG flight, partiti